In [1]:
# Import all functions from the required modules
from cordo_sherpa_module import *
from cordo_chimere_module import *
from expo_functions_module import *
from mortality_chimere_module import *
from association_module import *
from morbidity_chimere_module import *
print("Successfully loaded all modules")

loaded defined RR values
Successfully loaded all modules


In [2]:
# Paths to the files
path_fichier_shp = "data/2-output-data/donnees_shp"
title_shp = "donnees_insee_iris"
path_fichier_pourcents = "data/2-output-data"
title_pourcents = "pourcents"

# Load the concentration points
conc_points = coordo_sherpa(sc="s1", pol="ug_NO2", year=2019)

# Load the exported data
donnees_exportees = gpd.read_file(os.path.join(path_fichier_shp, f"{title_shp}.shp"))

# Transform the CRS of the exported data to match the concentration points
donnees_exportees_transformed = donnees_exportees.to_crs(epsg=conc_points.crs.to_epsg())

# Check if CRSs are the same
if conc_points.crs == donnees_exportees_transformed.crs:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are the same.")
else:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are different.")

Concentrations in 2019 and 2019 are calculated for the pollutant 'ug_no2' (s1).
CRS for conc_points_transformed and donnees_exportees_transformed are the same.


In [3]:
import os
# Define paths for shapefiles
path_fichier_shp = "data/2-output-data/donnees_shp"
path_fichier_shp_1 = "data/2-output-data/donnees_shp_1"
path_fichier_shp_2 = "data/2-output-data/donnees_shp_2"
path_fichier_shp_3 = "data/2-output-data/donnees_shp_3"
path_fichier_pourcents = "data/2-output-data"

# Titles for INSEE Data
title_shp = "donnees_insee_iris"
title_shp_1 = "donnees_insee_iris_toutage_1"
title_shp_2 = "donnees_insee_iris_toutage_2"
title_shp_3 = "donnees_insee_iris_toutage_3"
title_pourcents = "pourcents"

# Read shapefiles into GeoDataFrames
donnees_shp_1 = gpd.read_file(os.path.join(path_fichier_shp_1, f"{title_shp_1}.shp"))
donnees_shp_2 = gpd.read_file(os.path.join(path_fichier_shp_2, f"{title_shp_2}.shp"))
donnees_shp_3 = gpd.read_file(os.path.join(path_fichier_shp_3, f"{title_shp_3}.shp"))

# Combine the three GeoDataFrames
donnees_merged = gpd.GeoDataFrame(pd.concat([donnees_shp_1, donnees_shp_2, donnees_shp_3], ignore_index=True))
print(donnees_merged.head())

grille_combinee = gpd.read_file(os.path.join(path_fichier_pourcents, f"{title_pourcents}.shp"))
grille_combinee = grille_combinee.to_crs(conc_points.crs)

     iriscod irisname comcod    comname depcod   depname  regcod    regname  \
0  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
1  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
2  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
3  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
4  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   

   age   pop2019   pop2030   pop2050  mort2019  mort2030  mort2050  \
0    0  1.001248  1.028860  1.007603  0.003590  0.003423  0.002972   
1    1  1.024924  1.022945  1.015467  0.000575  0.000556  0.000491   
2    2  1.049225  1.018921  1.023012  0.000261  0.000246  0.000219   
3    3  1.078030  1.016966  1.029914  0.000157  0.000155  0.000142   
4    4  1.114215  1.017214  1.035872  0.000121  0.000118  0.000109   

                                            geometry  
0  POLYGON ((1052346.7 6848413, 1052502.3 6848474

In [4]:
#Merge the file with shared population file at comcod
mort_file_csv = r"data\Morbidity data check\morta30_new.csv"
mort_df = pd.read_csv(
    mort_file_csv,
    dtype=str,
    sep=";",
    engine="python",
    encoding="latin1"
)

# Strip whitespace from column names and values
mort_df.columns = mort_df.columns.str.strip().str.replace('"', '', regex=False)
for col in mort_df.columns:
    if mort_df[col].dtype == object:
        mort_df[col] = mort_df[col].str.strip().str.replace('"', '', regex=False)

# Rename and format codes
if 'COM_2018_CODE' in mort_df.columns:
    mort_df = mort_df.rename(columns={'COM_2018_CODE': 'comcod'})
    mort_df['comcod'] = mort_df['comcod'].str.upper().str.zfill(5)
else:
    raise ValueError("Column 'COM_2018_CODE' not found in mortality file.")

# Map arrondissements to their main communes at commune level (5-char codes)
arrondissement_mapping = {
    **{str(code).zfill(5): "75056" for code in range(75101, 75121)},  # Paris
    **{str(code).zfill(5): "13055" for code in range(13201, 13217)},  # Marseille
    **{str(code).zfill(5): "69123" for code in range(69381, 69390)},  # Lyon
}
mort_df['comcod'] = mort_df['comcod'].map(lambda x: arrondissement_mapping.get(x, x))
# Convert mortality column to numeric
if 'nb30_moy' in mort_df.columns:
    mort_df['morta_a30_moy'] = pd.to_numeric(
        mort_df['nb30_moy']
        .astype(str)
        .str.replace('\xa0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False),
        errors='coerce'
    ).fillna(0.0)
else:
    raise ValueError("Column 'nb30_moy' not found in mortality file.")

# --- File paths ---
population_file = "data/Morbidity data check/population_data.xlsx"
# --- Read population ---
if os.path.exists(population_file):
    pop_df = pd.read_excel(population_file, dtype={"insee": str}, engine="openpyxl")
else:
    raise FileNotFoundError(f"Population file not found: {population_file}")

pop_df = pop_df[["insee", "pop30p"]].rename(columns={"insee": "comcod"})
pop_df["comcod"] = pop_df["comcod"].str.upper().str.zfill(5)
pop_df["pop30p"] = pd.to_numeric(pop_df["pop30p"], errors="coerce").fillna(0)

# --- Merge mortality with population ---
commune_df = mort_df.merge(pop_df, on="comcod", how="left")

In [5]:
!pip install chardet
from pop_functions_module import *
# Paths to data files
path_contours = "data/1-processed-data/CONTOURS-IRIS"
path_iris = "data/1-processed-data/INSEE/donnees_iris.txt"
path_proj = "data/1-processed-data/INSEE/donnees_projections.txt"
path_depart = "data/1-processed-data/INSEE/num_depart.csv"
path_age_femmes = "data/1-processed-data/INSEE/donnees_age_femmes.csv"
path_age_hommes = "data/1-processed-data/INSEE/donnees_age_hommes.csv"
path_mortalite_hf = "data/1-processed-data/INSEE/donnees_deces_hf.csv"
#Distribute mortality and population of commune using function disaggregate_commune_by_age
# --- USAGE EXAMPLE ---
# commune_df = ... # (as before, with columns 'comcod', 'pop30p', 'morta_a30_moy')
# perc_hf = decomposition_age(age_hf) # age_hf is your departmental/France age pop table
#Load Departmental population for commune_age based disaggregation/distribution
donnees_geom = geometries(path_iris, path_depart, path_contours)
print("Top 2 rows of donnees_geom:")
print(donnees_geom.head())
age_hf = age_nat(path_age_femmes, path_age_hommes)
print("Top 2 rows of age_hf:")
print(age_hf.head())
perc_hf = decomposition_age(age_hf)
print("All columns of perc_hf:")
print(perc_hf.columns)
print("Top 2 rows of perc_hf:")
print(perc_hf.head())
mort_comm_age = disaggregate_commune_by_age(commune_df, perc_hf, year=2019, age_min=30)

print("Total mortality input:", commune_df["morta_a30_moy"].sum())
print("Total mortality after allocation:", mort_comm_age["morta_age"].sum())
print(mort_comm_age.head())

The INSEE data is extracted at the IRIS level.
Top 2 rows of donnees_geom:
  dep_cod           dep_name  region_cod                 region_name com_cod  \
0      67           Bas-Rhin        44.0                   Grand Est   67043   
1      13   Bouches-du-Rhône        93.0  Provence-Alpes-Côte d’Azur   13202   
2      56           Morbihan        53.0                    Bretagne   56185   
3      93  Seine-Saint-Denis        11.0               Ile-de-France   93063   
4      94       Val-de-Marne        11.0               Ile-de-France   94048   

                      com_name   iris_cod           iris_name  \
0                    Bischheim  670430101              Annexe   
1  Marseille 2e Arrondissement  132020101               Arenc   
2                       Quéven  561850101  B.A.N. Lann Bihoué   
3                  Romainville  930630101            Bas Pays   
4             Marolles-en-Brie  940480101  Bois de Notre-Dame   

   POP_2019_IRIS_TotAge                              

In [6]:
#WHO thresholds and commune based mortality function with RR of Sante Publique France and life table calculations for LE gain by age and overall (population-weighted)
RR_PM25 = 1.15      #1.15
RR_PM25_high = 1.25
RR_PM25_low = 1.05
RR_PM25_Hrapie = 1.10     #exposure range 5-70
RR_PM25_Hrapie_high = 1.13
RR_PM25_Hrapie_low = 1.06
RR_NO2 = 1.023
RR_NO2_high = 1.037
RR_NO2_low = 1.008
RR_NO2_Hrapie = 1.05    #exposure range 10-130
RR_NO2_Hrapie_high = 1.07
RR_NO2_Hrapie_low = 1.03
print(f'loaded defined RR values')

def mortalite_age_who_commune_new(mort_comm_age, donnees_expo, annee, pol):
    mort_comm_age = mort_comm_age.copy()
    donnees_expo = donnees_expo.copy()
    mort_comm_age["comcod"] = mort_comm_age["comcod"].astype(str).str.upper().str.zfill(5)
    donnees_expo["comcod"] = donnees_expo["comcod"].astype(str).str.upper().str.zfill(5)
    donnees_expo = donnees_expo.drop(columns="geometry", errors="ignore")

    df = pd.merge(mort_comm_age, donnees_expo[['comcod', 'meanconc']], on='comcod', how='left')
    df["meanconc"] = df["meanconc"].fillna(0)

    RR_dict_sensitivity = {
        "ug_PM25_RH50": (RR_PM25, RR_PM25_low, RR_PM25_high, 5),
        "ug_NO2": (RR_NO2, RR_NO2_low, RR_NO2_high, 10),
    }
    RR_dict_Main = {
    "ug_PM25_RH50": (RR_PM25_Hrapie, RR_PM25_Hrapie_low, RR_PM25_Hrapie_high, 5),
    "ug_NO2": (RR_NO2_Hrapie, RR_NO2_Hrapie_low, RR_NO2_Hrapie_high, 10)
    }
    RR, RR_low, RR_high, threshold = RR_dict_Main[pol]

    iris_interet = df[(df['age'] >= 30) & (df['age'] <= 99)].copy()
    iris_interet["pop_age"] = pd.to_numeric(iris_interet["pop_age"], errors="coerce").fillna(0)
    iris_interet["meanconc"] = pd.to_numeric(iris_interet["meanconc"], errors="coerce").fillna(0)

    weighted_conc = iris_interet.groupby("age")[["meanconc", "pop_age"]].apply(
        lambda x: np.average(x["meanconc"], weights=x["pop_age"]) if x["pop_age"].sum() > 0 else 0
    ).rename("pop_weighted_conc")

    iris_interet = iris_interet.merge(weighted_conc, left_on="age", right_index=True, how="left")
    iris_interet["meandelta"] = np.where(
        iris_interet["pop_weighted_conc"] > threshold,
        iris_interet["pop_weighted_conc"] - threshold,
        0
    )

    def avoided_mortality(rr):
        return iris_interet["morta_age"] * (1 - np.exp(-np.log(rr) * iris_interet["meandelta"] / 10))

    iris_interet["mortpol_age"] = avoided_mortality(RR).clip(lower=0)
    iris_interet["mortpol_age_LCI"] = avoided_mortality(RR_low).clip(lower=0)
    iris_interet["mortpol_age_UCI"] = avoided_mortality(RR_high).clip(lower=0)

    esp_dict = {2019: 80, 2030: 84, 2050: 86}
    esp = esp_dict.get(int(annee), 80)

    grouped = iris_interet.groupby("age").agg({
        "mortpol_age": "sum",
        "mortpol_age_LCI": "sum",
        "mortpol_age_UCI": "sum",
        "morta_age": "sum",
        "pop_age": "sum"
    }).reset_index()

    grouped["annees_gagnees"] = grouped["mortpol_age"] * (esp - grouped["age"])
    grouped["annees_gagnees_LCI"] = grouped["mortpol_age_LCI"] * (esp - grouped["age"])
    grouped["annees_gagnees_UCI"] = grouped["mortpol_age_UCI"] * (esp - grouped["age"])
    grouped["taux_initial"] = grouped["morta_age"] / grouped["pop_age"]
    grouped["taux_corrige"] = (grouped["morta_age"] - grouped["mortpol_age"]) / grouped["pop_age"]
    grouped["taux_corrige_LCI"] = (grouped["morta_age"] - grouped["mortpol_age_LCI"]) / grouped["pop_age"]
    grouped["taux_corrige_UCI"] = (grouped["morta_age"] - grouped["mortpol_age_UCI"]) / grouped["pop_age"]

    # --- LIFE TABLE CALCULATIONS ---
    ages = grouped["age"].to_numpy()
    taux_initial_arr = grouped["taux_initial"].to_numpy()
    taux_corrige_arr = grouped["taux_corrige"].to_numpy()
    taux_corrige_LCI_arr = grouped["taux_corrige_LCI"].to_numpy()
    taux_corrige_UCI_arr = grouped["taux_corrige_UCI"].to_numpy()
    pop_arr = grouped["pop_age"].to_numpy()

    # Cap the mortality rates at 1 for ages > 98
    taux_initial_arr = np.where(ages > 98, 1, taux_initial_arr)
    taux_corrige_arr = np.where(ages > 98, 1, taux_corrige_arr)
    taux_corrige_LCI_arr = np.where(ages > 98, 1, taux_corrige_LCI_arr)
    taux_corrige_UCI_arr = np.where(ages > 98, 1, taux_corrige_UCI_arr)

    le_initial = life_expectancy_table(ages, taux_initial_arr, interval=1, a_x=0.5, radix=100000)
    le_corrige = life_expectancy_table(ages, taux_corrige_arr, interval=1, a_x=0.5, radix=100000)
    le_corrige_LCI = life_expectancy_table(ages, taux_corrige_LCI_arr, interval=1, a_x=0.5, radix=100000)
    le_corrige_UCI = life_expectancy_table(ages, taux_corrige_UCI_arr, interval=1, a_x=0.5, radix=100000)

    grouped["LifeTable_LE_initial"] = le_initial
    grouped["LifeTable_LE_corrige"] = le_corrige
    grouped["LifeTable_LE_corrige_LCI"] = le_corrige_LCI
    grouped["LifeTable_LE_corrige_UCI"] = le_corrige_UCI

    # Life expectancy gain (months) per age
    grouped["LifeTable_LEgain_mo"] = (le_corrige - le_initial) * 12
    grouped["LifeTable_LEgain_mo_LCI"] = (le_corrige_LCI - le_initial) * 12
    grouped["LifeTable_LEgain_mo_UCI"] = (le_corrige_UCI - le_initial) * 12

    # --- OVERALL LIFE EXPECTANCY GAIN (months, population-weighted) ---
    total_pop = grouped["pop_age"].sum()

    avg_LE_initial = np.sum(le_initial * pop_arr) / total_pop
    avg_LE_corrige = np.sum(le_corrige * pop_arr) / total_pop
    avg_LE_corrige_LCI = np.sum(le_corrige_LCI * pop_arr) / total_pop
    avg_LE_corrige_UCI = np.sum(le_corrige_UCI * pop_arr) / total_pop

    overall_LEgain_mo = (avg_LE_corrige - avg_LE_initial) * 12
    overall_LEgain_mo_LCI = (avg_LE_corrige_LCI - avg_LE_initial) * 12
    overall_LEgain_mo_UCI = (avg_LE_corrige_UCI - avg_LE_initial) * 12

    grouped["avg_LE_initial"] = avg_LE_initial
    grouped["avg_LE_corrige"] = avg_LE_corrige
    grouped["avg_LE_corrige_LCI"] = avg_LE_corrige_LCI
    grouped["avg_LE_corrige_UCI"] = avg_LE_corrige_UCI

    grouped["overall_LifeTable_LEgain_mo"] = overall_LEgain_mo
    grouped["overall_LifeTable_LEgain_mo_LCI"] = overall_LEgain_mo_LCI
    grouped["overall_LifeTable_LEgain_mo_UCI"] = overall_LEgain_mo_UCI

    # --- Other summary columns ---
    tot_mort_30_99 = grouped["mortpol_age"].sum()
    tot_mort_30_99_LCI = grouped["mortpol_age_LCI"].sum()
    tot_mort_30_99_UCI = grouped["mortpol_age_UCI"].sum()
    baseline_mortality = grouped["morta_age"].sum()

    grouped["tot_mort_30_99"] = tot_mort_30_99
    grouped["tot_mort_30_99_LCI"] = tot_mort_30_99_LCI
    grouped["tot_mort_30_99_UCI"] = tot_mort_30_99_UCI
    grouped["baseline_mortality"] = baseline_mortality
    grouped["percent_avoided_mortality"] = (
                                                       tot_mort_30_99 / baseline_mortality) * 100 if baseline_mortality > 0 else np.nan
    grouped["percent_avoided_mortality_LCI"] = (
                                                           tot_mort_30_99_LCI / baseline_mortality) * 100 if baseline_mortality > 0 else np.nan
    grouped["percent_avoided_mortality_UCI"] = (
                                                           tot_mort_30_99_UCI / baseline_mortality) * 100 if baseline_mortality > 0 else np.nan
    return grouped.sort_values(by='age').reset_index(drop=True)


loaded defined RR values


In [7]:
# ----------------------------
# Life Table Function
# ----------------------------
def life_expectancy_table(age, m_x, interval=1, a_x=0.5, radix=100000):
    """
    Calculate life expectancy at each age using a simple life table.

    Parameters input:
        age (array-like): Ages.
        m_x (array-like): Mortality rates for each age.
        interval (int): Age interval (default 1).
        a_x : Fraction of interval lived by those who die. Default is 0.5.
        radix (int): Starting population (default 100,000).

    Returns:
        e_x (np.ndarray): Life expectancy at each age.
    """
    n = len(age)

    # Assign default value to `a_x` if it is None
    if a_x is None or isinstance(a_x, (float, int)):
        a_x = np.full(n, a_x)  # Create an array if scalar passed

    # Ensure mortality rate values are valid
    m_x = np.nan_to_num(m_x)  # Replace NaNs with 0
    m_x[m_x < 0] = 0          # No negative mortality rates
    m_x = np.clip(m_x, 0, 1)  # Clip mortality rates to [0, 1]

    # Probability of dying between ages x and x+1
    q_x = (interval * m_x) / (1 + (interval - a_x) * m_x)
    q_x = np.minimum(q_x, 1.0)

    # Compute l_x (probability of surviving to exact age)
    l_x = np.zeros(n)
    l_x[0] = radix
    for i in range(1, n):
        l_x[i] = l_x[i - 1] * (1 - q_x[i - 1])

    # Compute L_x (person-years lived in each interval)
    L_x = np.zeros(n)
    for i in range(n - 1):
        L_x[i] = interval * (l_x[i + 1] + a_x[i] * (l_x[i] - l_x[i + 1]))
    if m_x[-1] > 0:  # Avoid divide by zero for last L_x calculation
        L_x[-1] = l_x[-1] / m_x[-1]
    else:
        L_x[-1] = interval * l_x[-1]

    # Compute T_x (cumulative person-years lived above each age x)
    T_x = np.zeros(n)
    T_x[-1] = L_x[-1]
    for i in range(n - 2, -1, -1):
        T_x[i] = T_x[i + 1] + L_x[i]

    # Compute e_x (life expectancy at age x)
    e_x = T_x / np.where(l_x > 0, l_x, 1e-10)  # Ensure no division by zero

    return e_x

In [8]:
#Sante Publique France based 4 years average concentration from INERIS (2016-2019) for correction of mortality and morbidity later, with interpolation to 2019 grid and spatial join to commune shapefile for population-weighted concentration at commune level
import unicodedata
def normalize_column_to_utf8(df, column_name):
    """Normalize a specific column to ensure proper UTF-8 encoding."""
    if column_name in df.columns:
        df[column_name] = df[column_name].apply(
            lambda x: unicodedata.normalize('NFKD', str(x)).encode('ascii', 'ignore').decode('utf-8')
        )
    return df

##Code for getting average concentration from 2016-2019 for each point for correction later
from scipy.interpolate import griddata
def coordo_ineris_avg(pol):
    """
    Returns a GeoDataFrame of INERIS pollutant concentrations averaged over 2016–2019.
    All negative and -999 values are treated as missing (NaN) for both NO2 and PM2.5.
    All years are interpolated to the 2019 grid.
    """
    print("Starting coordo_ineris_avg function")
    pol = {
        "ug_PM25_RH50_high": "ug_PM25_RH50",
        "ug_PM25_RH50_low": "ug_PM25_RH50",
        "ug_NO2_high": "ug_NO2",
        "ug_NO2_low": "ug_NO2"
    }.get(pol, pol)

    var = "PM25" if pol == "ug_PM25_RH50" else "NO2"
    years = [2019]

    base_paths = {
        y: f"data/1-processed-data/SHERPA/conc-{y}/Reanalysed_FRA_{y}_{var}_avgannual_Ineris_v.Jan2024.nc"
        for y in years
    }

    path_2019 = base_paths[2019]
    if not os.path.exists(path_2019):
        raise FileNotFoundError(f"File not found: {path_2019}")

    with Dataset(path_2019) as nc19:
        lat_ref = nc19.variables['lat'][:]
        lon_ref = nc19.variables['lon'][:]
        ref_shape = nc19.variables[var][:].shape

    data_list = []
    for year in years:
        path = base_paths[year]
        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {path}")

        with Dataset(path) as nc:
            lat = nc.variables['lat'][:]
            lon = nc.variables['lon'][:]
            arr = nc.variables[var][:]
            conc = np.where((arr < 0) | (arr == -999), np.nan, arr)
            valid_cells = np.isfinite(conc).sum()
            print(f"Year {year}: valid grid cells = {valid_cells}, mean = {np.nanmean(conc):.3f}, min = {np.nanmin(conc):.3f}, max = {np.nanmax(conc):.3f}")

            # Interpolate all years to 2019 grid
            lon_grid, lat_grid = np.meshgrid(lon, lat)
            lon_ref_grid, lat_ref_grid = np.meshgrid(lon_ref, lat_ref)
            mask = np.isfinite(conc)
            if np.sum(mask) < 3:
                print(f"Warning: Too few valid points for interpolation in {year}. All values set to NaN.")
                conc_interp = np.full(lon_ref_grid.shape, np.nan)
            else:
                conc_interp = griddata(
                    np.column_stack((lon_grid.ravel()[mask.ravel()], lat_grid.ravel()[mask.ravel()])),
                    conc.ravel()[mask.ravel()],
                    (lon_ref_grid, lat_ref_grid),
                    method='linear'
                )
                nan_mask = np.isnan(conc_interp)
                if np.any(nan_mask):
                    nearest = griddata(
                        np.column_stack((lon_grid.ravel()[mask.ravel()], lat_grid.ravel()[mask.ravel()])),
                        conc.ravel()[mask.ravel()],
                        (lon_ref_grid, lat_ref_grid),
                        method='nearest'
                    )
                    conc_interp[nan_mask] = nearest[nan_mask]
            print(f"Year {year}: interpolated valid cells = {np.isfinite(conc_interp).sum()}")
            data_list.append(conc_interp)

    conc_stack = np.stack(data_list, axis=0)

    with np.errstate(invalid='ignore'):
        meanconc = np.nanmean(conc_stack, axis=0)

    print(f"Final meanconc: min = {np.nanmin(meanconc):.3f}, max = {np.nanmax(meanconc):.3f}, mean = {np.nanmean(meanconc):.3f}, valid cells = {np.isfinite(meanconc).sum()}")

    lon_grid, lat_grid = np.meshgrid(lon_ref, lat_ref)
    df = pd.DataFrame({
        'x': lon_grid.ravel(),
        'y': lat_grid.ravel(),
        'meanconc': meanconc.ravel()
    })
    conc_ineris = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y), crs="EPSG:4326")

    print("Finished coordo_ineris_avg function")
    return conc_ineris
#Merge conc_ineris with commune shapefile using spatial join
def merge_conc_with_communes(conc_ineris):
    """
    Merge concentration points with commune shapefile using spatial join.
    Keeps population and INSEE code, returns commune-level averages.
    Returns a GeoDataFrame with columns: 'comcod', 'meanconc', 'geometry'.
    The simple mean of commune means is also printed for reference.
    """
    shp_path = "data/1-processed-data/CONTOURS-COMM/COMMUNE.shp"
    communes = gpd.read_file(shp_path)

    # Make sure CRS matches (reproject if needed)
    if communes.crs != conc_ineris.crs:
        communes = communes.to_crs(conc_ineris.crs)

    # Spatial join: assign each concentration point to a commune
    points_with_commune = gpd.sjoin(conc_ineris, communes, how="left", predicate="within")

    # Aggregate mean concentration per commune (simple average of points within each commune)
    conc_by_commune = (
        points_with_commune.groupby("INSEE_COM")["meanconc"]
        .mean()
        .reset_index()
    )
    # Merge back with commune polygons
    communes_with_conc = communes.merge(conc_by_commune, on="INSEE_COM", how="left")
    communes_with_conc.rename(columns={"INSEE_COM": "comcod"}, inplace=True)
    communes_with_conc = communes_with_conc[["comcod", "meanconc", "geometry"]]
    #rename conc19 to meanconc for consistency with other datasets
    #communes_with_conc.rename(columns={"conc19": "meanconc"}, inplace=True)
    # Print simple mean of commune means
    simple_mean = communes_with_conc["meanconc"].mean()
    print(f"Simple mean of commune means: {simple_mean:.4f}")
    return communes_with_conc

In [9]:
#Function to run mortality avoided and life expectancy at population-weighted commune conc
import os
import logging
import warnings

# Suppress warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

# Configure logging for the main script
logging.basicConfig(
    level=logging.DEBUG, format="%(asctime)s - %(levelname)s - %(message)s"
)

# Define scenarios, pollutants, and years
scenarios = ["s1"]
pollutants = ["ug_NO2", "ug_PM25_RH50"]  # e.g., ["ug_NO2", "ug_PM25_RH50"] #run one-by-one to avoid memory issues
years = ["2019"]

def process_combination(
    params, donnees_exportees_transformed, grille_combinee, donnees_merged
):
    scenario, year, pollutant = params
    logging.info(f"Processing combination: Scenario={scenario}, Pollutant={pollutant}, Year={year}")

    try:
        # Load computation data of 2019
        logging.info("Starting coordo_ineris function")
        conc_chimere = coordo_ineris_avg(pollutant)
        conc_chimere = conc_chimere.to_crs(donnees_exportees_transformed.crs)
        logging.info("Finished processing coordo_ineris function")
        donnees_expo = merge_conc_with_communes(conc_chimere)
        logging.info("Exposure processing completed")

        # Ensure output directories
        output_path = os.path.join("data", "2-output-data", scenario, pollutant, year)
        os.makedirs(output_path, exist_ok=True)

        # Pass data into the HIA function
        result = mortalite_age_who_commune_new(
            mort_comm_age=mort_comm_age, donnees_expo=donnees_expo, annee=year, pol=pollutant
        )
        # Save result
        result.to_csv(os.path.join(output_path, "mortality_WHO_main.csv"), index=False)
        logging.info(f"Result exported successfully for Scenario={scenario}")

    except Exception as e:
        logging.error(f"An Error at Scenario={scenario}, Pollutant={pollutant}")
        logging.exception(f"Error details: {e}")

# Main execution
if __name__ == "__main__":
    # Align CRS of shapefiles to the CRS of donnees_exportees_transformed
    target_crs = donnees_exportees_transformed.crs
    donnees_exportees_transformed = donnees_exportees_transformed.to_crs(target_crs)
    grille_combinee = grille_combinee.to_crs(target_crs)

    # Prepare parameters for processing
    params_list = [
        (scenario, year, pollutant) for scenario in scenarios for year in years for pollutant in pollutants
    ]
    # Process each combination of scenario, year, and pollutant
    for params in params_list:
        process_combination(params, donnees_exportees_transformed, grille_combinee, donnees_merged)

    logging.info("Successfully processed all combinations.")

INFO:root:Processing combination: Scenario=s1, Pollutant=ug_NO2, Year=2019
INFO:root:Starting coordo_ineris function


Starting coordo_ineris_avg function
Year 2019: valid grid cells = 533248, mean = 7.079, min = 0.091, max = 34.021
Year 2019: interpolated valid cells = 1644201
Final meanconc: min = 0.091, max = 34.021, mean = 8.218, valid cells = 1644201


INFO:root:Finished processing coordo_ineris function


Finished coordo_ineris_avg function


INFO:root:Exposure processing completed


...

INFO:root:Finished processing coordo_ineris function


Finished coordo_ineris_avg function


INFO:root:Exposure processing completed


Simple mean of commune means: 8.5714


INFO:root:Result exported successfully for Scenario=s1
INFO:root:Successfully processed all combinations.


In [93]:
#Distribute health data by age to the main projected mortality file (for morbidity analysis)
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Load health data
health_data = pd.read_csv(r"C:\Users\aysharma\Eqis_codes\data\Morbidity data check\final_health_data.csv")
# Keep INSEE as 5-character commune codes (baseline)
health_data['insee'] = health_data['insee'].astype(str).str.upper().str.zfill(5)

# Preprocess population and mortality data
df = donnees_merged.copy()

# Use 5-character commune codes as baseline from donnees_merged
df['comcod'] = df['comcod'].astype(str).str.upper().str.zfill(5)

# Map arrondissements to their main communes at IRIS level (5-char codes)
arrondissement_mapping = {
    **{str(code).zfill(5): "75056" for code in range(75101, 75121)},  # Paris
    **{str(code).zfill(5): "13055" for code in range(13201, 13217)},  # Marseille
    **{str(code).zfill(5): "69123" for code in range(69381, 69390)},  # Lyon
}
df['comcod'] = df['comcod'].map(lambda x: arrondissement_mapping.get(x, x))

# Diagnostics: commune coverage baseline vs health data
pop_communes = df['comcod'].dropna().unique()
health_communes = health_data['insee'].dropna().unique()

print(f"Baseline unique communes in donnees_merged (5-char): {len(pop_communes)}")
print(f"Unique communes in health_data (5-char): {len(health_communes)}")

pop_only = set(pop_communes) - set(health_communes)
health_only = set(health_communes) - set(pop_communes)
print(f"Communes present only in population: {len(pop_only)}")
print(f"Communes present only in health: {len(health_only)}")

# Merge IRIS×age population with commune health totals ---
# Keep baseline comcod universe from donnees_merged (LEFT join)
donnees_merged = df.merge(
    health_data[['insee', 'disease', 'absolute_incidence', 'inc_100000']],
    left_on="comcod",
    right_on="insee",
    how="left"
)

print(f"Communes retained after merge (based on donnees_merged): {donnees_merged['comcod'].nunique()}")

# Define disease-specific age ranges
disease_age_groups = {
    "Hypertension": (18, 99),
    "Lung Cancer": (35, 99),
    "Asthma in children": (0, 17),
    "Asthma in adult": (18, 39),
    "COPD": (40, 99),
    "ALRI in children": (0, 12),
    "Stroke": (35, 99),
    "IHD events": (30, 99),
    "Diabetes T2": (45, 99),
}

# --- Allocate commune totals to IRIS×age ---
donnees_merged["absolute_incidence_iris"] = 0.0

for disease, (amin, amax) in disease_age_groups.items():
    mask = (donnees_merged["disease"] == disease) & (donnees_merged["age"].between(amin, amax))
    # population denominator per commune
    denom = donnees_merged.loc[mask].groupby(["comcod", "disease"])["pop2019"].transform("sum")
    share = donnees_merged.loc[mask, "pop2019"] / denom.replace(0, np.nan)
    donnees_merged.loc[mask, "absolute_incidence_iris"] = (
            share.fillna(0) * donnees_merged.loc[mask, "absolute_incidence"].fillna(0)
    )

# --- Verification ---
print("\nDisease-specific totals before and after distribution:")
print("-" * 80)
print(f"{'Disease':<20} {'Before':>15} {'After':>15} {'Difference':>15} {'Relative %':>15}")
print("-" * 80)

for disease in disease_age_groups.keys():
    before = health_data.loc[health_data["disease"] == disease, "absolute_incidence"].sum()
    after = donnees_merged.loc[donnees_merged["disease"] == disease, "absolute_incidence_iris"].sum()
    diff = after - before
    rel = (diff / before * 100) if before > 0 else 0
    print(f"{disease:<20} {before:>15.2f} {after:>15.2f} {diff:>15.2f} {rel:>15.2f}")

total_before = health_data["absolute_incidence"].sum()
total_after = donnees_merged ["absolute_incidence_iris"].sum()
diff = total_after - total_before
rel = (diff / total_before * 100) if total_before > 0 else 0
print("-" * 80)
print(f"{'TOTAL':<20} {total_before:>15.2f} {total_after:>15.2f} {diff:>15.2f} {rel:>15.2f}")

Baseline unique communes in donnees_merged (5-char): 33787
Unique communes in health_data (5-char): 35228
Communes present only in population: 0
Communes present only in health: 1441
Communes retained after merge (based on donnees_merged): 33787

Disease-specific totals before and after distribution:
--------------------------------------------------------------------------------
Disease                       Before           After      Difference      Relative %
--------------------------------------------------------------------------------
Hypertension               712414.50       697848.75       -14565.75           -2.04
Lung Cancer                 39634.80        38760.66         -874.14           -2.21
Asthma in children         201989.25       198422.25        -3567.00           -1.77
Asthma in adult             92322.25        90681.50        -1640.75           -1.78
COPD                       194491.25       190100.75        -4390.50           -2.26
ALRI in children          

In [103]:
#Morbidity Function with WHO thresholds for baseline comparison
SPF_THRESHOLDS = {"ug_PM25_RH50": 5, "ug_NO2": 10}
#Main function with mc simulation for propagation of uncertainties
def morbidity_mortality_mc_by_age_spf_comcod(
        donnees_merged_new,
        donnees_expo,
        annee,
        pol,
        disease,
        morb_config,
        n_mc=1000,
        seed=42,
        chunk_size=10000
):
    """
    Monte Carlo morbidity + mortality HIA
    - Optimized for memory: Uses vectorized operations and minimizes intermediate large object creation.
    """

    import numpy as np
    import pandas as pd
    import logging
    import unicodedata

    logger = logging.getLogger(__name__)
    rng = np.random.default_rng(seed)
    annee = int(annee)

    if "comcod" not in donnees_merged_new.columns:
        return pd.DataFrame({"error": ["Missing comcod"]})

    pop_col = f"pop{annee}" if f"pop{annee}" in donnees_merged_new.columns else "pop2019"
    mort_col = f"mort{annee}" if f"mort{annee}" in donnees_merged_new.columns else "mort2019"

    # Optimization: Filter by disease immediately to reduce data size
    def norm(s):
        if not isinstance(s, str): return ""
        return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower().replace(" ", "_").replace(
            "-", "_")

    target_disease_norm = norm(disease)

    # Efficiently filter the population dataframe first
    df_pop = donnees_merged_new.loc[
        donnees_merged_new["disease"].apply(norm) == target_disease_norm,
        ["comcod", "iriscod", "disease", "age", pop_col, mort_col, "absolute_incidence_iris"]
    ].copy()

    if df_pop.empty:
        return pd.DataFrame()

    df_pop["comcod"] = df_pop["comcod"].astype(str).str.upper()
    df_pop[pop_col] = pd.to_numeric(df_pop[pop_col], errors="coerce").fillna(0.0)
    df_pop[mort_col] = pd.to_numeric(df_pop[mort_col], errors="coerce").fillna(0.0)
    df_pop["absolute_incidence_iris"] = pd.to_numeric(df_pop["absolute_incidence_iris"], errors="coerce").fillna(0.0)

    # Exposure aggregation
    if "meanconc" not in donnees_expo.columns:
        return pd.DataFrame({"error": ["Missing meanconc"]})

    if "comcod" not in donnees_expo.columns and "iriscod" in donnees_expo.columns:
        tmp = donnees_expo[["iriscod", "meanconc"]].merge(
            df_pop[["iriscod", "comcod", pop_col]], on="iriscod", how="inner"
        )
        tmp["comcod"] = tmp["comcod"].astype(str).str.upper()
        tmp["weighted_conc"] = tmp["meanconc"] * tmp[pop_col]
        grp = tmp.groupby("comcod")
        df_expo_comcod = (grp["weighted_conc"].sum() / grp[pop_col].sum().replace(0, np.nan)).fillna(0).reset_index()
        df_expo_comcod.columns = ["comcod", "meanconc"]
    else:
        df_expo_comcod = (
            donnees_expo[["comcod", "meanconc"]]
            .assign(comcod=lambda d: d["comcod"].astype(str).str.upper())
            .groupby("comcod", as_index=False)["meanconc"]
            .mean()
        )

    merged = df_pop.merge(df_expo_comcod, on="comcod", how="left")
    merged["meanconc"] = pd.to_numeric(merged["meanconc"], errors="coerce").fillna(0.0)

    # Use absolute delta vs threshold so outputs are absolute (not negative benefits)
    threshold = SPF_THRESHOLDS.get(pol, 0.0)
    merged["meandelta"] = (merged["meanconc"] - float(threshold)).clip(lower=0)

    cfg, rr_key = find_matching_morbidity_config(disease, pol, morb_config)
    if cfg is None:
        return pd.DataFrame()

    RRm, RRm_lo, RRm_hi = cfg[rr_key]
    dw_m, dw_lo, dw_hi = cfg["disability_weight"]
    min_age = int(cfg.get("pm25_age_min", cfg.get("no2_age_min", 0)))
    max_age = int(cfg.get("pm25_age_max", cfg.get("no2_age_max", 99)))

    merged["age"] = pd.to_numeric(merged["age"], errors="coerce")
    merged = merged[(merged["age"] >= min_age) & (merged["age"] <= max_age)]

    if merged.empty:
        return pd.DataFrame()

    # Pre-calculate sampling distributions
    beta_morb_central = np.log(RRm)
    esp = {2019: 82, 2030: 83, 2050: 85}.get(annee, 82)
    duration = cfg.get("duration", 1.0)

    # Aggregate by age first to minimize loop iterations
    merged["pop_x_delta"] = merged["meandelta"] * merged[pop_col]
    age_agg = merged.groupby("age").agg({
        "pop_x_delta": "sum",
        pop_col: "sum",
        "absolute_incidence_iris": "sum",
        mort_col: "sum"
    })
    age_agg["meandelta"] = (age_agg["pop_x_delta"] / age_agg[pop_col].replace(0, np.nan)).fillna(0)

    rows = []
    for age, data in age_agg.iterrows():
        delta = data["meandelta"]
        b_cases = data["absolute_incidence_iris"]
        rem_life = max(esp - age, 0)
        eff_dur = min(duration, rem_life)

        # Central
        AF_morb_c = 1 - np.exp(-beta_morb_central * delta / 10)

        # Initialize Monte Carlo accumulators for percentiles
        all_cases_mc = []
        all_yld_mc = []

        # Process MC in chunks
        for i in range(0, n_mc, chunk_size):
            current_chunk = min(chunk_size, n_mc - i)

            beta_morb_chunk = rng.normal(np.log(RRm), (np.log(RRm_hi) - np.log(RRm_lo)) / 3.92, current_chunk)
            dw_samples_chunk = np.clip(rng.normal(dw_m, (dw_hi - dw_lo) / 3.92, current_chunk), 0, None)

            AF_morb_chunk = 1 - np.exp(-beta_morb_chunk * delta / 10)
            cases_chunk = b_cases * AF_morb_chunk

            all_cases_mc.append(cases_chunk)
            all_yld_mc.append(cases_chunk * dw_samples_chunk * eff_dur)

        cases_mc = np.concatenate(all_cases_mc)
        yld_mc = np.concatenate(all_yld_mc)

        rows.append({
            "age": age,
            "avoided_cases_central": b_cases * AF_morb_c,
            "avoided_cases_low": np.percentile(cases_mc, 2.5),
            "avoided_cases_high": np.percentile(cases_mc, 97.5),
            "YLD_central": (b_cases * AF_morb_c) * dw_m * eff_dur,
            "YLD_low": np.percentile(yld_mc, 2.5),
            "YLD_high": np.percentile(yld_mc, 97.5)
        })

    donnees = pd.DataFrame(rows)
    donnees[["disease", "pollutant", "year", "mc_iterations"]] = [disease, pol, annee, n_mc]
    return donnees.sort_values("age").reset_index(drop=True)

In [105]:
# -------------------------------------------------
# Morbidity runner function: DALY, YLD, YLL, avoided cases (with MC simulation for propogation of uncertainties)
# -------------------------------------------------
warnings.simplefilter(action='ignore', category=FutureWarning)
logging.basicConfig(level=logging.DEBUG,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Run PM only (do not inherit an old NO2 value from globals)
scenarios = ["s1"]
pollutants = ["ug_NO2"]
annees = ["2019"]


def process_combination_morb(params, donnees_merged, donnees_exportees_transformed, grille_combinee):
    sc, annee, pol = params
    logging.info(f"[MORB] Processing Scenario={sc}, Pollutant={pol}, Year={annee}")
    try:
        # --- Concentrations ---
        # Load computation data of 2019
        logging.info("Starting coordo_ineris function")
        conc_chimere = coordo_ineris_avg(pol)
        conc_chimere = conc_chimere.to_crs(donnees_exportees_transformed.crs)
        logging.info("Finished processing coordo_ineris function")
        donnees_expo = merge_conc_with_communes(conc_chimere)
        logging.info("Exposure processing completed")

        # --- Output directory ---
        path_fichier_expo = os.path.join("data", "2-output-data", sc, pol, annee)
        os.makedirs(path_fichier_expo, exist_ok=True)

        # --- Run morbidity for each disease ---
        all_results = []
        for cfg in morb_config:
            disease = cfg["disease"]
            df_morb = morbidity_mortality_mc_by_age_spf_comcod(
                donnees_merged,
                donnees_expo,
                annee,
                pol,
                disease,
                morb_config,
                n_mc=1000,
                seed=42,
                chunk_size=10000
            )

            if df_morb is None or df_morb.empty:
                logging.warning(f"[MORB] No results for {disease}")
                continue

            df_morb["scenario"] = sc
            all_results.append(df_morb)

        # --- Save combined deterministic output ---
        if all_results:
            out_df = pd.concat(all_results, ignore_index=True)
            out_file = os.path.join(path_fichier_expo, "morbidity_results_WHO.csv")
            out_df.to_csv(out_file, index=False)
            logging.info(f"[MORB] Saved deterministic results: {out_file}")

    except Exception as e:
        logging.exception(f"[MORB] Error Scenario={sc}, Pol={pol}, Year={annee}: {e}")


# -------------------------------------------------
# Main execution
# -------------------------------------------------
if __name__ == "__main__":
    target_crs = donnees_exportees_transformed.crs
    donnees_exportees = donnees_exportees.to_crs(target_crs)
    grille_combinee = grille_combinee.to_crs(target_crs)
    params_list = [(sc, annee, pol) for sc in scenarios for annee in annees for pol in pollutants]
    for params in params_list:
        process_combination_morb(params, donnees_merged, donnees_exportees, grille_combinee)
    logging.info("✅ Morbidity processing finished.")


INFO:root:[MORB] Processing Scenario=s1, Pollutant=ug_NO2, Year=2019
INFO:root:Starting coordo_ineris function


Starting coordo_ineris_avg function
Year 2016: valid grid cells = 533248, mean = 8.304, min = 0.098, max = 38.236
Year 2016: interpolated valid cells = 1644201
Year 2017: valid grid cells = 533248, mean = 7.917, min = 0.068, max = 37.900
Year 2017: interpolated valid cells = 1644201
Year 2018: valid grid cells = 533248, mean = 7.442, min = 0.111, max = 36.115
Year 2018: interpolated valid cells = 1644201
Year 2019: valid grid cells = 533248, mean = 7.079, min = 0.091, max = 34.021
Year 2019: interpolated valid cells = 1644201
Final meanconc: min = 0.150, max = 36.292, mean = 8.839, valid cells = 1644201


INFO:root:Finished processing coordo_ineris function


Finished coordo_ineris_avg function


INFO:root:Exposure processing completed


Simple mean of commune means: 8.5064


INFO:root:[MORB] Saved deterministic results: data\2-output-data\s1\ug_NO2\2019\morbidity_results_WHO.csv
INFO:root:✅ Morbidity processing finished.
